In [1]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.12.11 environment at: C:\projects\End-to-End-LLMOPS-Project\.venv
Resolved 67 packages in 520ms
Prepared 6 packages in 2.76s
Uninstalled 1 package in 308ms
Installed 13 packages in 3.27s
 + coloredlogs==15.0.1
 + flatbuffers==25.9.23
 + humanfriendly==10.0
 + mpmath==1.3.0
 - numpy==2.3.3
 + numpy==2.2.6
 + onnxruntime==1.23.0
 + opencv-python==4.12.0.88
 + pillow==11.3.0
 + pyclipper==1.3.0.post6
 + pyreadline3==3.5.4
 + rapidocr-onnxruntime==1.4.4
 + shapely==2.1.2
 + sympy==1.14.0


In [2]:
import os

from dotenv import load_dotenv

load_dotenv()


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [3]:
#Data Ingestion

In [4]:
from langchain.document_loaders import TextLoader

In [5]:
loader = TextLoader("/Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/Agentic AI.txt", encoding="utf8")
documents = loader.load()

RuntimeError: Error loading /Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/Agentic AI.txt

In [ ]:
documents[0].page_content[:500]  # Print the first 500 characters of the first documen

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [7]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [ ]:
text_chunks=text_splitter.split_documents(documents)

In [ ]:
text_chunks

In [ ]:
! uv pip install faiss-cpu

In [ ]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

In [ ]:
embeddings=OpenAIEmbeddings()

In [ ]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [ ]:
vectorstore

In [ ]:
retriever=vectorstore.as_retriever()

In [ ]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

In [ ]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [ ]:
prompt=ChatPromptTemplate.from_template(template)

In [ ]:
prompt

In [ ]:
from langchain.schema.output_parser import StrOutputParser

In [ ]:
output_parser=StrOutputParser()

In [ ]:
from langchain.chat_models import ChatOpenAI

llm_model=ChatOpenAI(model_name="gpt-4o-mini")

In [ ]:
from langchain.schema.runnable import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [ ]:
rag_chain.invoke("tell me about Agentic AI")